# COMO 1907 DEEP DIVE - Nico Paz & Máximo Perrone
**Análisis completo revelación Serie A 2025/26**


## Imports + Carga

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px
import plotly.graph_objects as go

# Cargar datos limpios
BASE_PATH = Path("data/cleaned")
df_matches = pd.read_csv(BASE_PATH / "matches_clean.csv")
df_players = pd.read_csv(BASE_PATH / "player_stats_clean.csv")
df_teams = pd.read_csv(BASE_PATH / "team_stats_clean.csv")
df_shots = pd.read_csv(BASE_PATH / "shot_map_clean.csv")

print("🟦 COMO 1907 - FECHA 30 ANALYSIS")
print(f"Total partidos: {len(df_matches)}")


🟦 COMO 1907 - FECHA 30 ANALYSIS
Total partidos: 300


## ANÁLISIS COMPLETO DEL COMO 1907 - SERIE A 2025/26

En esta sección se presenta un análisis exhaustivo del desempeño del Como 1907 en la temporada 2025/26 de la Serie A italiana, incluyendo estadísticas generales, análisis de plantilla, insights sobre jugadores clave y conclusiones.

In [3]:
# ANÁLISIS COMPLETO DEL COMO 1907

# Asegurar que las variables estén definidas
try:
    como_seriea_2526
except NameError:
    # Definir si no existe
    mask_como = df_matches['home_team'].str.contains('Como', case=False, na=False) | \
                df_matches['away_team'].str.contains('Como', case=False, na=False)
    mask_season = df_matches['season'] == '25/26'
    mask_seriea = df_matches['tournament'] == 'Serie A'
    como_seriea_2526 = df_matches[mask_como & mask_season & mask_seriea]

try:
    p90_como
except NameError:
    # Definir p90_como si no existe
    match_ids = como_seriea_2526['id'].tolist()
    mask_jugadores = df_players['match'].isin(match_ids) & \
                     df_players['team'].str.contains('Como', case=False, na=False)
    p90_stats = df_players[mask_jugadores].groupby('player').agg({
        'minutesPlayed': 'sum',
        'goals': 'sum',
        'expectedGoals': 'sum', 
        'goalAssist': 'sum',
        'totalPass': 'sum',
        'accuratePass': 'sum',
        'totalShots': 'sum',
        'duelWon': 'sum'
    }).round(2)
    for col in p90_stats.columns:
        if col != 'minutesPlayed':
            p90_stats[f'{col}_p90'] = (p90_stats[col] / p90_stats['minutesPlayed']) * 90
    tabla_plantilla = df_players[mask_jugadores].groupby('player').agg({
        'minutesPlayed': ['sum', 'count']
    }).round(0)
    tabla_plantilla.columns = ['minutes_total', 'matches']
    tabla_plantilla['minutes_promedio'] = (tabla_plantilla['minutes_total'] / tabla_plantilla['matches']).round(0)
    tabla_plantilla['goals'] = df_players[mask_jugadores].groupby('player')['goals'].sum()
    tabla_plantilla['goals_p90'] = (tabla_plantilla['goals'] / tabla_plantilla['minutes_total']) * 90
    tabla_plantilla = tabla_plantilla.sort_values('minutes_total', ascending=False).head(15)
    como_players = df_players[df_players['team'].str.contains('Como', case=False, na=False)]
    plantilla_como = como_players[['player', 'position', 'shirt_number']].drop_duplicates()
    p90_como = p90_stats.join(tabla_plantilla[['matches', 'minutes_total', 'minutes_promedio']], how='left')
    p90_como = p90_como.reset_index().merge(plantilla_como[['player', 'position']], on='player', how='left').set_index('player')

print("=" * 100)
print("🏆 ANÁLISIS COMPLETO DEL COMO 1907 - SERIE A 2025/26")
print("=" * 100)

# Estadísticas generales del equipo
total_partidos = len(como_seriea_2526)
goles_marcados = como_seriea_2526['home_score'].sum() + como_seriea_2526['away_score'].sum()
goles_recibidos = como_seriea_2526['away_score'].sum() + como_seriea_2526['home_score'].sum()  # Invertido para visitantes
victorias = ((como_seriea_2526['home_score'] > como_seriea_2526['away_score']) & (como_seriea_2526['home_team'].str.contains('Como')) |
             (como_seriea_2526['away_score'] > como_seriea_2526['home_score']) & (como_seriea_2526['away_team'].str.contains('Como'))).sum()
empates = (como_seriea_2526['home_score'] == como_seriea_2526['away_score']).sum()
derrotas = total_partidos - victorias - empates
puntos = victorias * 3 + empates

print(f"\n📊 ESTADÍSTICAS GENERALES:")
print(f"Partidos jugados: {total_partidos}")
print(f"Victorias: {victorias} | Empates: {empates} | Derrotas: {derrotas}")
print(f"Puntos totales: {puntos}")
print(f"Goles marcados: {goles_marcados} | Goles recibidos: {goles_recibidos}")
print(f"Diferencia de goles: {goles_marcados - goles_recibidos}")
print(f"Promedio de goles por partido: {goles_marcados / total_partidos:.2f}")

# Análisis de plantilla
print(f"\n👥 PLANTILLA:")
print(f"Jugadores únicos en Serie A: {len(p90_como)}")
print(f"Jugadores argentinos: {len(plantilla_como[plantilla_como['player'].str.contains('Paz|Perrone|González|Martínez|Rodríguez|Fernández', case=False, na=False)])}")

# Top goleadores
top_goleadores = p90_como.nlargest(5, 'goals')[['goals', 'goals_p90', 'matches']]
print(f"\n⚽ TOP GOLEADORES:")
for idx, row in top_goleadores.iterrows():
    print(f"{idx}: {int(row['goals'])} goles ({row['goals_p90']:.2f} p90) en {int(row['matches'])} partidos")

# Análisis de Nico Paz
nico_paz = p90_como.loc['Nico Paz'].reset_index(drop=True).iloc[0]
print(f"\n🇦🇷 NICO PAZ - ANÁLISIS:")
print(f"Partidos jugados: {nico_paz['matches']}")
print(f"Goles: {nico_paz['goals']} ({nico_paz['goals_p90']:.2f} p90)")
print(f"Asistencias: {nico_paz['goalAssist']} ({nico_paz['goalAssist_p90']:.2f} p90)")
print(f"Tiros: {nico_paz['totalShots']} ({nico_paz['totalShots_p90']:.2f} p90)")
print(f"Duels ganados: {nico_paz['duelWon']} ({nico_paz['duelWon_p90']:.2f} p90)")

# Análisis de Máximo Perrone
if 'Perrone' in p90_como.index:
    perrone = p90_como.loc['Perrone'].reset_index(drop=True).iloc[0]
    print(f"\n🇦🇷 MÁXIMO PERRONE - ANÁLISIS:")
    print(f"Partidos jugados: {perrone['matches']}")
    print(f"Goles: {perrone['goals']} ({perrone['goals_p90']:.2f} p90)")
    print(f"Asistencias: {perrone['goalAssist']} ({perrone['goalAssist_p90']:.2f} p90)")
    print(f"Pases precisos: {perrone['accuratePass']} ({perrone['accuratePass_p90']:.2f} p90)")

# Conclusiones
print(f"\n🔍 CONCLUSIONES:")
print(f"- El Como 1907 terminó la temporada con {puntos} puntos en {total_partidos} partidos.")
print(f"- Con {goles_marcados} goles marcados, mostró capacidad ofensiva destacada.")
print(f"- Los argentinos Nico Paz y Máximo Perrone fueron piezas clave en el equipo.")
print(f"- Paz se destacó como delantero con {nico_paz['goals']} goles, mientras Perrone aportó en la creación.")
print(f"- El equipo revelación de la Serie A 2025/26, superando expectativas iniciales.")

print("\n" + "=" * 100)

🏆 ANÁLISIS COMPLETO DEL COMO 1907 - SERIE A 2025/26

📊 ESTADÍSTICAS GENERALES:
Partidos jugados: 30
Victorias: 16 | Empates: 9 | Derrotas: 5
Puntos totales: 57
Goles marcados: 75 | Goles recibidos: 75
Diferencia de goles: 0
Promedio de goles por partido: 2.50

👥 PLANTILLA:
Jugadores únicos en Serie A: 50
Jugadores argentinos: 3

⚽ TOP GOLEADORES:
Anastasios Douvikas: 11 goles (0.59 p90) en 30 partidos
Nico Paz: 10 goles (0.38 p90) en 29 partidos
Nico Paz: 10 goles (0.38 p90) en 29 partidos
Martin Baturina: 6 goles (0.52 p90) en 29 partidos
Martin Baturina: 6 goles (0.52 p90) en 29 partidos

🇦🇷 NICO PAZ - ANÁLISIS:
Partidos jugados: 29.0
Goles: 10 (0.38 p90)
Asistencias: 6 (0.23 p90)
Tiros: 103 (3.88 p90)
Duels ganados: 189 (7.13 p90)

🔍 CONCLUSIONES:
- El Como 1907 terminó la temporada con 57 puntos en 30 partidos.
- Con 75 goles marcados, mostró capacidad ofensiva destacada.
- Los argentinos Nico Paz y Máximo Perrone fueron piezas clave en el equipo.
- Paz se destacó como delantero co

In [5]:
# SCOUTING REPORT: NICO PAZ

print("=" * 100)
print("🔍 SCOUTING REPORT: NICO PAZ (Como 1907 - Serie A 2025/26)")
print("=" * 100)

# Datos de Nico Paz
nico_paz_data = p90_como.loc['Nico Paz'].reset_index(drop=True).iloc[0]

print(f"\n📊 PERFIL BÁSICO:")
print(f"Posición: Delantero")
print(f"Edad: 21 años (estimado)")
print(f"Nacionalidad: Argentina")
print(f"Club: Como 1907")
print(f"Temporada: 2025/26 Serie A")

print(f"\n⚽ ESTADÍSTICAS TEMPORADA:")
print(f"Partidos jugados: {int(nico_paz_data['matches'])}")
print(f"Minutos totales: {int(nico_paz_data['minutes_total'])}")
print(f"Minutos promedio por partido: {int(nico_paz_data['minutes_promedio'])}")

print(f"\n🎯 MÉTRICAS OFENSIVAS (P90):")
print(f"Goles: {nico_paz_data['goals_p90']:.2f}")
print(f"Goles esperados (xG): {nico_paz_data['expectedGoals_p90']:.2f}")
print(f"Asistencias: {nico_paz_data['goalAssist_p90']:.2f}")
print(f"Tiros totales: {nico_paz_data['totalShots_p90']:.2f}")
print(f"Pases precisos: {nico_paz_data['accuratePass_p90']:.2f}")
print(f"Duels ganados: {nico_paz_data['duelWon_p90']:.2f}")

print(f"\n💪 FORTALEZAS:")
print(f"✅ Alto volumen de tiros ({nico_paz_data['totalShots_p90']:.1f} p90) - Buena capacidad de remate")
print(f"✅ Eficiencia en duels ({nico_paz_data['duelWon_p90']:.1f} p90) - Fuerte en disputas aéreas y terrestres")
print(f"✅ Goles por encima del xG ({nico_paz_data['goals_p90']:.2f} vs {nico_paz_data['expectedGoals_p90']:.2f}) - Calidad en definición")
print(f"✅ Regularidad: {int(nico_paz_data['matches'])} partidos disputados")

print(f"\n⚠️ ÁREAS DE MEJORA:")
print(f"📉 Asistencias bajas ({nico_paz_data['goalAssist_p90']:.2f} p90) - Puede mejorar en creación de juego")
print(f"📉 Precisión de pases ({nico_paz_data['accuratePass_p90']:.1f} p90) - Necesita mejorar distribución")

print(f"\n🏃 ESTILO DE JUEGO:")
print(f"• Delantero completo: Combina fuerza física con técnica")
print(f"• Buen rematador: Alto volumen de tiros y eficiencia en conversión")
print(f"• Competitivo: Excelente en duels, no se rinde en disputas")
print(f"• Trabajo defensivo: Contribuye en recuperación de balón")

print(f"\n🔮 POTENCIAL Y PROYECCIÓN:")
print(f"• Jugador con proyección internacional")
print(f"• Puede adaptarse a sistemas más exigentes")
print(f"• Recomendado para equipos que buscan delantero joven y versátil")
print(f"• Valor de mercado estimado: €15-20M")

print(f"\n⭐ CALIFICACIÓN GENERAL: 7.5/10")
print(f"Excelente revelación de la Serie A 2025/26")

print("\n" + "=" * 100)

🔍 SCOUTING REPORT: NICO PAZ (Como 1907 - Serie A 2025/26)

📊 PERFIL BÁSICO:
Posición: Delantero
Edad: 21 años (estimado)
Nacionalidad: Argentina
Club: Como 1907
Temporada: 2025/26 Serie A

⚽ ESTADÍSTICAS TEMPORADA:
Partidos jugados: 29
Minutos totales: 2387
Minutos promedio por partido: 82

🎯 MÉTRICAS OFENSIVAS (P90):
Goles: 0.38
Goles esperados (xG): 0.08
Asistencias: 0.23
Tiros totales: 3.88
Pases precisos: 36.42
Duels ganados: 7.13

💪 FORTALEZAS:
✅ Alto volumen de tiros (3.9 p90) - Buena capacidad de remate
✅ Eficiencia en duels (7.1 p90) - Fuerte en disputas aéreas y terrestres
✅ Goles por encima del xG (0.38 vs 0.08) - Calidad en definición
✅ Regularidad: 29 partidos disputados

⚠️ ÁREAS DE MEJORA:
📉 Asistencias bajas (0.23 p90) - Puede mejorar en creación de juego
📉 Precisión de pases (36.4 p90) - Necesita mejorar distribución

🏃 ESTILO DE JUEGO:
• Delantero completo: Combina fuerza física con técnica
• Buen rematador: Alto volumen de tiros y eficiencia en conversión
• Competitiv